In [1]:
from bs4 import BeautifulSoup, NavigableString
import openai
import os
import re
import html


In [2]:
def translate(prompt):
        # Get OpenAI API Key from environment variable
        api_key = os.environ["OPENAI_API_KEY"]

        client = openai.OpenAI(
            api_key=api_key,
        )
        response = client.chat.completions.create(
            messages=[
                {
                    "role": "system",
                    "content": "Translate the text provided by the user to English, while keeping the original structure of the text. Be especially careful with the formulas not to break them. Do not omit anything."
                },
                {
                    "role": "user",
                    "content": prompt,
                }
            ],
            model="gpt-4-1106-preview",
        )
        
        return response.choices[0].message.content

In [3]:
def replace_images_with_markdown(problem_soup):
    images = problem_soup.find_all('img')
    for idx, img in enumerate(images, start=1):
        img.replace_with(f"![image]({idx}.png)")
    return problem_soup

In [4]:
def rewrite_formula_and_lists(soup):
    for sub_tag in soup.find_all('sub'):
        sub_tag.replace_with('_{' + sub_tag.text + '}')

    for sup_tag in soup.find_all('sup'):
        sup_tag.replace_with('^{' + sup_tag.text + '}')

    for var_tag in soup.find_all('var'):
        var_tag.replace_with('$' + var_tag.text + '$')

    for li_tag in soup.find_all('li'):
        li_tag.replace_with('- ' + li_tag.text)
    return soup

In [8]:
import os
import json
import tqdm


def load_problems(problems_folder):
    """
    Scan the folder to retrieve problems with data.
    """
    subdirs = [os.path.join(problems_folder, d) for d in os.listdir(problems_folder) 
               if os.path.isdir(os.path.join(problems_folder, d))]
    return subdirs

def load_problem_data(folder_path):
    """
    Load the data.json file from the specified folder path.
    """
    with open(os.path.join(folder_path, 'data.json'), 'r') as file:
        data = json.load(file)
    return data

def save_new_data(folder_path, new_data):
    """
    Save the modified data to a new file in the folder path.
    """
    with open(os.path.join(folder_path, 'new_data.json'), 'w') as file:
        json.dump(new_data, file, indent=4)

def process_problem(folder_path, active_prefixes=None):
    prefix_to_function = {
        'ac': "process_atcoder",
        'az': "process_aizu",
        'cc': "process_codechef",
        'cf': "process_codeforces",
        'cw': "process_codewars",
        'ep': "process_euler",
        'g4g': "process_geeksforgeeks",
        'hr': "process_hackerrank",
        'lc': "process_leetcode",
        'ok': "process_openkattis"
    }

    # If no specific prefixes are specified, all are considered active
    if active_prefixes is None:
        active_prefixes = prefix_to_function.keys()

    prefix = os.path.basename(folder_path).split('_')[0]
    if prefix in prefix_to_function and prefix in active_prefixes:
        globals()[prefix_to_function[prefix]](folder_path)
    elif prefix not in active_prefixes:
        pass
    else:
        print(f"Unknown prefix: {prefix}")

In [6]:
def process_codeforces(folder_path):
    data = load_problem_data(folder_path)
    new_data = {} 
    new_data.update(data)
    # Rewrite sample tests
    sample_tests = new_data.pop("sample_tests")
    new_sample_tests = {"inputs": [], "outputs": []}
    for item in sample_tests:
        new_sample_tests["inputs"].append(item["input"])
        new_sample_tests["outputs"].append(item["output"])
    new_data["sample_tests"] = json.dumps(new_sample_tests)
    # Question text
    raw_problem = new_data.pop("problem_raw")
    new_data["raw_problem"] = raw_problem
    soup = BeautifulSoup(raw_problem, 'html.parser')
    replace_images_with_markdown(soup)
    #   Remove header
    soup.find('div', class_='header').decompose()
    rewrite_formula_and_lists(soup)
    #   Insert a newline after each paragraph tag
    for p in soup.find_all(['p', 'div', 'pre']):
        p.append(soup.new_string('\n'))
    # Replace <br> with newline characters
    for br in soup.find_all("br"):
        br.replace_with("\n")
    
    new_data["question"] = soup.text
    
    
    save_new_data(folder_path, new_data)

In [7]:
def process_codewars(folder_path):
    data = load_problem_data(folder_path)
    crawled_meta = data.pop("crawled_meta")
    url = crawled_meta["url"]
    raw_problem = crawled_meta["raw_problem"]
    new_data = {}  # Modify this as per your logic
    new_data.update(data)
    new_data["raw_problem"] = raw_problem
    new_data["question"] = replace_images_with_markdown(BeautifulSoup(raw_problem, "html.parser")).text
    new_data["url"] = url
    save_new_data(folder_path, new_data)

In [8]:
def process_euler(folder_path):
    data = load_problem_data(folder_path)
    data.pop("problem")
    raw_problem = data["raw_problem"]
    new_data = {}  # Modify this as per your logic
    new_data.update(data)
    soup = BeautifulSoup(raw_problem, 'html.parser')
    replace_images_with_markdown(soup)
    rewrite_formula_and_lists(soup)
    new_data["question"] = soup.text
    save_new_data(folder_path, new_data)

In [9]:
def process_geeksforgeeks(folder_path):
    data = load_problem_data(folder_path)
    crawled_meta = data.pop("crawled_meta")
    url = crawled_meta["url"]
    raw_problem = crawled_meta["raw_problem"]
    new_data = {}  # Modify this as per your logic
    new_data.update(data)
    new_data["raw_problem"] = raw_problem
    soup = BeautifulSoup(raw_problem, 'html.parser')
    replace_images_with_markdown(soup)
    rewrite_formula_and_lists(soup)
    new_data["question"] = soup.text
    new_data["url"] = url
    save_new_data(folder_path, new_data)


In [30]:
def hr_insert_image_marks(clean_question: str, str_w_img: str):
    # Find all occurrences of image markdown in raw_s
    pattern = r'(\S+)?\s*(!\[image\]\(.*?\))\s*(\S+)?'
    matches = re.finditer(pattern, str_w_img)

    # Process each markdown
    for match in matches:
        before_markdown, markdown, after_markdown = match.groups()



        if before_markdown is None:
            # The markdown is at the beginning
            clean_question = markdown + "\n" + clean_question
            continue

        if after_markdown is None:
            # The markdown is at the end
            clean_question = clean_question + "\n" + markdown + '\n'
            continue
        
        # The markdown is in the middle of the text
        # Find the position of the word before the markdown in s
        before_index = clean_question.find(before_markdown)
        if before_index == -1:
            before_index = 0
        # Find the position of the word after the markdown in s
        after_index = clean_question.find(after_markdown, before_index)
        if after_index == -1:
            after_index = len(clean_question)

        # Insert the markdown at the appropriate position
        insert_index = after_index

        clean_question = clean_question[:insert_index] + "\n" + markdown + "\n" + clean_question[insert_index:]

    return clean_question

def process_hackerrank(folder_path):
    data = load_problem_data(folder_path)
    crawled_meta = data.pop("crawled_meta")
    raw_problem = crawled_meta["raw_problem"]
    soup = BeautifulSoup(raw_problem, 'html.parser')
    replace_images_with_markdown(soup)
    rewrite_formula_and_lists(soup)
    problem_w_img_label = soup.text
    clean_question = data["question"]
    
    new_data = {}
    new_data.update(data)
    new_data["raw_problem"] = raw_problem
    new_data["question"] = hr_insert_image_marks(clean_question, problem_w_img_label)
    save_new_data(folder_path, new_data)

In [11]:
def process_leetcode(folder_path):
    data = load_problem_data(folder_path)
    raw_problem = data.pop("crawled_meta")["raw_problem_data"]["data"]["question"]["content"]

    new_data = {}  # Modify this as per your logic
    new_data.update(data)
    new_data["raw_problem"] = raw_problem
    soup = BeautifulSoup(raw_problem, 'html.parser')
    replace_images_with_markdown(soup)
    rewrite_formula_and_lists(soup)
    new_data["question"] = soup.text
    save_new_data(folder_path, new_data)

In [12]:
def ok_clean_text(soup):
    replace_images_with_markdown(soup)
    rewrite_formula_and_lists(soup)

    # Remove '\n' in the text, except in <table> tags
    # Find all text elements
    for text in soup.find_all(text=True):
        if isinstance(text, NavigableString):
            # Check if any parent of this text is a <table> tag
            if not any(parent.name == 'table' for parent in text.parents):
                # Replace \n with a space
                updated_text = re.sub(r'(\n+|\s\s+)', ' ', text)
                updated_text = re.sub(r' +', ' ', updated_text)
                text.replace_with(updated_text.strip())

    # Adjust the order of sample inputs and outputs
    for table in soup.find_all('table', class_="sample"):
        trs = table.find_all('tr')
        assert len(trs) == 2
        tr_head, tr_body = trs[0], trs[1]
        head_ths = tr_head.find_all('th')
        assert len(head_ths) == 2
        th_input, th_output = head_ths[0].text.strip(), head_ths[1].text.strip()

        tds = tr_body.find_all('td')
        assert len(tds) == 2
        td_input, td_output = tds[0].text.strip(), tds[1].text.strip()
        
        new_str = f"{th_input}\n{td_input}\n\n\n{th_output}\n{td_output}\n\n\n"

        table.replace_with(new_str)


    for h2_tag in soup.find_all('h2'):
        # Insert a newline before each <h2> tag
        h2_tag.insert_before('\n')

    for tag in soup.find_all(['p', 'h2', 'ul', 'li']):
        tag.append(soup.new_string('\n'))
    
    return soup.text

def process_openkattis(folder_path):
    new_data = {}

    data = load_problem_data(folder_path)
    crawled_data = data.pop("crawled_meta")
    raw_problem = crawled_data["raw_problem"]
    soup = BeautifulSoup(raw_problem, 'html.parser')
    cleaned_question = ok_clean_text(soup)
    
    new_data.update(data)
    new_data["name"] = crawled_data["problem_name"]
    new_data["raw_problem"] = raw_problem

    new_data["question"] = cleaned_question
    save_new_data(folder_path, new_data)

In [13]:
def ac_convert_tags_within_pre(pre_block):
    new_contents = []
    for content in pre_block.contents:
        if isinstance(content, NavigableString):
            if '\n' in content:
                # Split the string and re-insert it with newlines
                parts = content.split('\n')
                for part in parts:
                    new_contents.append(NavigableString(part))
                    new_contents.append(NavigableString('\n'))
            else:
                new_contents.append(content)
        else:
            # Apply tag replacements for other tags like <var>, <sub>, and <sup>
            if content.name == 'var':
                new_contents.append('$' + content.text + '$')
            elif content.name == 'sub':
                new_contents.append('_{' + content.text + '}')
            elif content.name == 'sup':
                new_contents.append('^{' + content.text + '}')
    
    # Clear the original contents and add the new ones
    pre_block.clear()
    for new_content in new_contents:
        pre_block.append(new_content)
        
def ac_convert_tags_and_extract_text(html_src):
    global translated_problems
    soup = BeautifulSoup(html_src, 'html.parser')

    # Add line breaks after each <h3> tag
    for h3 in soup.find_all('h3'):
        h3.insert_after(NavigableString('\n'))

    # Handle <pre> blocks separately
    pre_blocks = soup.find_all('pre')
    for pre_block in pre_blocks:
        ac_convert_tags_within_pre(pre_block)
    
    # Handle the remaining tags
    rewrite_formula_and_lists(soup)

    # Search for elements with class 'lang-en'
    lang_en = soup.find(class_='lang-en')
    if lang_en:
        replace_images_with_markdown(lang_en)
        result = lang_en.text
    else:
        # If no 'lang-en' found, search for 'lang-jp'
        lang_jp = soup.find(class_='lang-jp')
        if lang_jp:
            jp_text = lang_jp.text
            result = translate(jp_text)
        
    
    result = re.sub(r'\n{2,}', '\n\n', result)

    # Use a regular expression to find "Problem Statement" case-insensitively
    match = re.search("problem statement", result, re.IGNORECASE)
    assert match is not None  # Ensure that "Problem Statement" was found
    result = result[match.end():].strip()
    result = html.unescape(result)
    
    return result

def process_atcoder(folder_path):
    data = load_problem_data(folder_path)
    raw_problem = data.pop("crawled_meta")["raw_problem"]
    new_data = {}
    new_data.update(data)
    new_data["raw_problem"] = raw_problem

    # Parse problem text
    new_data["question"] = ac_convert_tags_and_extract_text(raw_problem)

    save_new_data(folder_path, new_data)

### CodeChef

In [14]:
def process_codechef(folder_path):
    data = load_problem_data(folder_path)
    crawled_meta = data.pop("crawled_meta")
    raw_problem = crawled_meta["raw_problem"]
    new_data = {}  # Modify this as per your logic
    new_data.update(data)
    new_data["raw_problem"] = raw_problem
    soup = BeautifulSoup(raw_problem, 'html.parser')
    replace_images_with_markdown(soup)
    rewrite_formula_and_lists(soup)
    new_data["question"] = soup.text
    save_new_data(folder_path, new_data)

### AIZU

In [15]:
def contains_japanese(text):
    """
    Check if the input text contains any Japanese characters.
    This includes kanji, hiragana, and katakana.

    Args:
    text (str): The text to check.

    Returns:
    bool: True if Japanese characters are found, False otherwise.
    """
    # Japanese character ranges
    # Hiragana: 3040-309F, Katakana: 30A0-30FF, Kanji: 4E00-9FAF
    for character in text:
        if '\u3040' <= character <= '\u309F' or \
           '\u30A0' <= character <= '\u30FF' or \
           '\u4E00' <= character <= '\u9FAF':
            return True
    return False

def az_convert_tags_within_pre(pre_block):
    new_contents = []
    for content in pre_block.contents:
        if isinstance(content, NavigableString):
            if '\n' in content:
                # Split the string and re-insert it with newlines
                parts = content.split('\n')
                for part in parts:
                    new_contents.append(NavigableString(part))
                    new_contents.append(NavigableString('\n'))
            else:
                new_contents.append(content)
        else:
            # Apply tag replacements for other tags like <var>, <sub>, and <sup>
            if content.name == 'var':
                new_contents.append('$' + content.text + '$')
            elif content.name == 'sub':
                new_contents.append('_{' + content.text + '}')
            elif content.name == 'sup':
                new_contents.append('^{' + content.text + '}')
    
    # Clear the original contents and add the new ones
    pre_block.clear()
    for new_content in new_contents:
        pre_block.append(new_content)
        
def az_convert_tags_and_extract_text(html_src):
    global translated_problems
    soup = BeautifulSoup(html_src, 'html.parser')
    replace_images_with_markdown(soup)

    # Add line breaks after each <h3> tag
    for h3 in soup.find_all('h3'):
        h3.insert_after(NavigableString('\n'))

    # Handle <pre> blocks separately
    pre_blocks = soup.find_all('pre')
    for pre_block in pre_blocks:
        az_convert_tags_within_pre(pre_block)
    
    # Handle the remaining tags
    for sub_tag in soup.find_all('sub'):
        sub_tag.replace_with('_{' + sub_tag.text + '}')

    for sup_tag in soup.find_all('sup'):
        sup_tag.replace_with('^{' + sup_tag.text + '}')

    for var_tag in soup.find_all('var'):
        var_tag.replace_with('$' + var_tag.text + '$')

    for li_tag in soup.find_all('li'):
        li_tag.replace_with('- ' + li_tag.text)


    def count_markdown_tags(text):
        """Count the specific Markdown tags formatted like ![image]({idx}.png)."""
        return len(re.findall(r'!\[image\]\(\{.*?\}\.png\)', text))

    # Initialize a counter for the loop
    counter = 0
    max_trials = 10

    # Extract text and count original Markdown tags
    original_text = soup.text
    original_tag_count = count_markdown_tags(original_text)

    # Loop with a maximum of 10 trials
    if contains_japanese(original_text):
        # initial trial
        translated_text = translate(original_text, temperature=0.0)
        translated_tag_count = count_markdown_tags(translated_text)
        if original_tag_count != translated_tag_count:
            # Try multiple times with a higher temperature
            while counter < max_trials:
                translated_text = translate(original_text, temperature=0.6)
                translated_tag_count = count_markdown_tags(translated_text)
                counter += 1
                if original_tag_count == translated_tag_count:
                    break
        result = translated_text
    else:
        result = original_text



    result = re.sub(r'\n{2,}', '\n\n', result)

    result = html.unescape(result)
    
    return result

def process_aizu(folder_path):
    data = load_problem_data(folder_path)
    raw_problem = data["raw_problem"]
    new_data = {}
    new_data.update(data)

    # Parse problem text
    new_data["question"] = az_convert_tags_and_extract_text(raw_problem)

    save_new_data(folder_path, new_data)


# Run

In [9]:
problems = load_problems("/home/kaixin/Desktop/mmcode/mmcode_dataset")
active_prefixes = ['cf']

for problem in tqdm.tqdm(problems):
    process_problem(problem, active_prefixes=active_prefixes)

  0%|          | 2/3548 [00:00<00:01, 2609.21it/s]


TypeError: string indices must be integers, not 'str'